In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("continuallearningexperiments.csv").dropna()
metrics = ['test_abs_rel', 'test_sq_rel', 'test_log10', 'test_rmse', 'test_rmse_log', 'test_a1', 'test_a2', 'test_a3']

In [2]:
df_full = df[df['test_region_type'] == 'full'].drop(columns=['test_region_type'])
#df_full

In [3]:
def compute_difference_proportions(group):
    before_adaptation_group = group[group['adaptation_state'] == 'before_adaptation']
    before_adaptation_row = {metric: None for metric in metrics}
    if len(before_adaptation_group) > 0:
        before_adaptation_row = before_adaptation_group.iloc[0]
    after_adaptation_group = group[group['adaptation_state'] == 'after_adaptation']
    after_adaptation_row = {metric: None for metric in metrics}
    if len(after_adaptation_group) > 0:
        after_adaptation_row = after_adaptation_group.iloc[0]
    differences = {}
    for metric in metrics:
        differences[metric] = None
        if before_adaptation_row[metric] and after_adaptation_row[metric]:
            differences[metric] = (before_adaptation_row[metric] - after_adaptation_row[metric]) / before_adaptation_row[metric]
    return pd.Series(differences)

In [4]:
df_diff_prop = df_full.groupby(['source_train_dataset', 'target_train_dataset', 'training_method', 'test_dataset'], as_index=False).apply(compute_difference_proportions)
df_diff_prop

,source_train_dataset,target_train_dataset,training_method,test_dataset,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
0,DDAD,KITTI,BOFedSCDepth,DDAD,-0.055814,0.030673,0.000000,0.030499,-0.002625,0.004878,-0.003628,0.000000
1,DDAD,KITTI,BOFedSCDepth,KITTI,0.644860,0.771613,0.588710,0.380131,0.484594,-0.799587,-0.206767,-0.056867
2,DDAD,KITTI,BOFedSCDepth(ConstrainedLoss),DDAD,-0.074419,-0.024760,-0.035398,0.001143,-0.041995,0.014634,0.002418,0.008830
3,DDAD,KITTI,BOFedSCDepth(ConstrainedLoss),KITTI,0.629283,0.756452,0.580645,0.371440,0.476190,-0.783058,-0.205514,-0.056867
4,DDAD,KITTI,BOFedSCDepth(ConstrainedLossRetrain),DDAD,-0.009302,0.088507,0.044248,0.058537,0.036745,-0.037398,-0.016929,-0.008830
5,DDAD,KITTI,BOFedSCDepth(ConstrainedLossRetrain),KITTI,0.641745,0.758065,0.596774,0.389089,0.484594,-0.797521,-0.206767,-0.056867
6,DDAD,KITTI,BOFedSCDepth(Retrain),DDAD,-0.009302,0.076312,0.044248,0.055944,0.028871,-0.037398,-0.018138,-0.006623
7,DDAD,KITTI,BOFedSCDepth(Retrain),KITTI,0.635514,0.754516,0.588710,0.384945,0.481793,-0.795455,-0.206767,-0.056867
8,DDAD,KITTI,FedSCDepth,DDAD,0.412088,0.425285,0.532751,0.173622,0.505391,-0.481651,-0.364668,-0.306152
9,DDAD,KITTI,FedSCDepth,KITTI,0.810510,0.944021,0.856338,0.627537,0.827298,-1.728707,-1.053305,-0.800731


In [5]:
df_diff_prop_avg = df_diff_prop.groupby(['source_train_dataset', 'target_train_dataset', 'training_method'], as_index=False)[metrics].mean()
df_diff_prop_avg

,source_train_dataset,target_train_dataset,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
0,DDAD,KITTI,BOFedSCDepth,0.294523,0.401143,0.294355,0.205315,0.240985,-0.397354,-0.105197,-0.028433
1,DDAD,KITTI,BOFedSCDepth(ConstrainedLoss),0.277432,0.365846,0.272623,0.186291,0.217098,-0.384212,-0.101548,-0.024018
2,DDAD,KITTI,BOFedSCDepth(ConstrainedLossRetrain),0.316221,0.423286,0.320511,0.223813,0.260670,-0.417460,-0.111848,-0.032848
3,DDAD,KITTI,BOFedSCDepth(Retrain),0.313106,0.415414,0.316479,0.220444,0.255332,-0.416426,-0.112452,-0.031745
4,DDAD,KITTI,FedSCDepth,0.611299,0.684653,0.694545,0.400579,0.666344,-1.105179,-0.708986,-0.553441
5,DDAD,KITTI,FedSCDepth(Average&Retrain),0.609348,0.659536,0.684402,0.371683,0.653077,-1.093136,-0.699816,-0.544858
6,DDAD,KITTI,FedSCDepth(Average),0.613827,0.695375,0.697503,0.406032,0.670597,-1.102310,-0.713593,-0.556303
7,DDAD,KITTI,FedSCDepth(Retrain),0.608551,0.691357,0.690953,0.404871,0.666554,-1.090557,-0.711162,-0.554872
8,DDAD,KITTI,SCDepth,0.036284,0.083559,-0.074964,0.067660,-0.040555,-0.217916,-0.049283,-0.001531
9,KITTI,DDAD,BOFedSCDepth,-0.115644,-0.665821,0.004918,-0.108650,-0.008312,-0.122116,-0.038906,-0.023072


In [6]:
df_diff_prop_avg_ddad_to_kitti = df_diff_prop_avg[df_diff_prop_avg['source_train_dataset'] == 'KITTI']
df_diff_prop_avg_ddad_to_kitti.sort_values(by=['test_abs_rel'],ascending=False)

,source_train_dataset,target_train_dataset,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
16,KITTI,DDAD,FedSCDepth(Retrain),-0.064024,-0.128648,-0.008990,0.021059,0.026531,-0.084938,-0.036399,-0.024845
9,KITTI,DDAD,BOFedSCDepth,-0.115644,-0.665821,0.004918,-0.108650,-0.008312,-0.122116,-0.038906,-0.023072
12,KITTI,DDAD,BOFedSCDepth(Retrain),-0.125717,-0.521183,-0.017362,-0.066966,-0.001498,-0.111248,-0.041513,-0.024595
14,KITTI,DDAD,FedSCDepth(Average&Retrain),-0.156301,-0.475017,-0.049328,-0.049825,0.003350,-0.094509,-0.041787,-0.028623
11,KITTI,DDAD,BOFedSCDepth(ConstrainedLossRetrain),-0.197765,-0.846700,-0.053726,-0.144280,-0.038653,-0.099509,-0.035452,-0.021091
10,KITTI,DDAD,BOFedSCDepth(ConstrainedLoss),-0.203692,-0.899101,-0.050522,-0.152969,-0.039278,-0.109064,-0.036363,-0.022300
13,KITTI,DDAD,FedSCDepth,-0.229573,-0.761978,-0.102955,-0.135676,-0.048933,-0.087109,-0.031085,-0.023390
15,KITTI,DDAD,FedSCDepth(Average),-0.267175,-1.002261,-0.123671,-0.190020,-0.064945,-0.085324,-0.030833,-0.023092
17,KITTI,DDAD,SCDepth,-0.467646,-1.958495,-0.238991,-0.400340,-0.184426,-0.033856,0.003523,-0.004965


In [7]:
df_diff_prop_avg_kitti_to_ddad = df_diff_prop_avg[df_diff_prop_avg['source_train_dataset'] == 'DDAD']
df_diff_prop_avg_kitti_to_ddad.sort_values(by=['test_abs_rel'],ascending=False)

,source_train_dataset,target_train_dataset,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
6,DDAD,KITTI,FedSCDepth(Average),0.613827,0.695375,0.697503,0.406032,0.670597,-1.102310,-0.713593,-0.556303
4,DDAD,KITTI,FedSCDepth,0.611299,0.684653,0.694545,0.400579,0.666344,-1.105179,-0.708986,-0.553441
5,DDAD,KITTI,FedSCDepth(Average&Retrain),0.609348,0.659536,0.684402,0.371683,0.653077,-1.093136,-0.699816,-0.544858
7,DDAD,KITTI,FedSCDepth(Retrain),0.608551,0.691357,0.690953,0.404871,0.666554,-1.090557,-0.711162,-0.554872
2,DDAD,KITTI,BOFedSCDepth(ConstrainedLossRetrain),0.316221,0.423286,0.320511,0.223813,0.260670,-0.417460,-0.111848,-0.032848
3,DDAD,KITTI,BOFedSCDepth(Retrain),0.313106,0.415414,0.316479,0.220444,0.255332,-0.416426,-0.112452,-0.031745
0,DDAD,KITTI,BOFedSCDepth,0.294523,0.401143,0.294355,0.205315,0.240985,-0.397354,-0.105197,-0.028433
1,DDAD,KITTI,BOFedSCDepth(ConstrainedLoss),0.277432,0.365846,0.272623,0.186291,0.217098,-0.384212,-0.101548,-0.024018
8,DDAD,KITTI,SCDepth,0.036284,0.083559,-0.074964,0.067660,-0.040555,-0.217916,-0.049283,-0.001531


In [8]:
df_diff_prop_avg_ciclic = df_diff_prop_avg.groupby(['training_method'], as_index=False)[metrics].mean()

In [9]:
df_diff_prop_avg_ciclic.sort_values(by=['test_abs_rel'],ascending=False)

,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
7,FedSCDepth(Retrain),0.272264,0.281354,0.340981,0.212965,0.346543,-0.587747,-0.373780,-0.289859
5,FedSCDepth(Average&Retrain),0.226523,0.092260,0.317537,0.160929,0.328214,-0.593822,-0.370802,-0.286740
4,FedSCDepth,0.190863,-0.038663,0.295795,0.132452,0.308706,-0.596144,-0.370036,-0.288416
6,FedSCDepth(Average),0.173326,-0.153443,0.286916,0.108006,0.302826,-0.593817,-0.372213,-0.289697
3,BOFedSCDepth(Retrain),0.093695,-0.052885,0.149558,0.076739,0.126917,-0.263837,-0.076983,-0.028170
0,BOFedSCDepth,0.089440,-0.132339,0.149636,0.048332,0.116336,-0.259735,-0.072052,-0.025753
2,BOFedSCDepth(ConstrainedLossRetrain),0.059228,-0.211707,0.133393,0.039766,0.111008,-0.258484,-0.073650,-0.026970
1,BOFedSCDepth(ConstrainedLoss),0.036870,-0.266628,0.111051,0.016661,0.088910,-0.246638,-0.068955,-0.023159
8,SCDepth,-0.215681,-0.937468,-0.156977,-0.166340,-0.112490,-0.125886,-0.022880,-0.003248


In [10]:
df_diff_prop_avg_ciclic.sort_values(by=['test_sq_rel'],ascending=False)

,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
7,FedSCDepth(Retrain),0.272264,0.281354,0.340981,0.212965,0.346543,-0.587747,-0.373780,-0.289859
5,FedSCDepth(Average&Retrain),0.226523,0.092260,0.317537,0.160929,0.328214,-0.593822,-0.370802,-0.286740
4,FedSCDepth,0.190863,-0.038663,0.295795,0.132452,0.308706,-0.596144,-0.370036,-0.288416
3,BOFedSCDepth(Retrain),0.093695,-0.052885,0.149558,0.076739,0.126917,-0.263837,-0.076983,-0.028170
0,BOFedSCDepth,0.089440,-0.132339,0.149636,0.048332,0.116336,-0.259735,-0.072052,-0.025753
6,FedSCDepth(Average),0.173326,-0.153443,0.286916,0.108006,0.302826,-0.593817,-0.372213,-0.289697
2,BOFedSCDepth(ConstrainedLossRetrain),0.059228,-0.211707,0.133393,0.039766,0.111008,-0.258484,-0.073650,-0.026970
1,BOFedSCDepth(ConstrainedLoss),0.036870,-0.266628,0.111051,0.016661,0.088910,-0.246638,-0.068955,-0.023159
8,SCDepth,-0.215681,-0.937468,-0.156977,-0.166340,-0.112490,-0.125886,-0.022880,-0.003248


In [11]:
df_diff_prop_avg_ciclic.sort_values(by=['test_rmse'],ascending=False)

,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
7,FedSCDepth(Retrain),0.272264,0.281354,0.340981,0.212965,0.346543,-0.587747,-0.373780,-0.289859
5,FedSCDepth(Average&Retrain),0.226523,0.092260,0.317537,0.160929,0.328214,-0.593822,-0.370802,-0.286740
4,FedSCDepth,0.190863,-0.038663,0.295795,0.132452,0.308706,-0.596144,-0.370036,-0.288416
6,FedSCDepth(Average),0.173326,-0.153443,0.286916,0.108006,0.302826,-0.593817,-0.372213,-0.289697
3,BOFedSCDepth(Retrain),0.093695,-0.052885,0.149558,0.076739,0.126917,-0.263837,-0.076983,-0.028170
0,BOFedSCDepth,0.089440,-0.132339,0.149636,0.048332,0.116336,-0.259735,-0.072052,-0.025753
2,BOFedSCDepth(ConstrainedLossRetrain),0.059228,-0.211707,0.133393,0.039766,0.111008,-0.258484,-0.073650,-0.026970
1,BOFedSCDepth(ConstrainedLoss),0.036870,-0.266628,0.111051,0.016661,0.088910,-0.246638,-0.068955,-0.023159
8,SCDepth,-0.215681,-0.937468,-0.156977,-0.166340,-0.112490,-0.125886,-0.022880,-0.003248


In [12]:
df_diff_prop_avg_ciclic.sort_values(by=['test_a1'],ascending=True)

,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
4,FedSCDepth,0.190863,-0.038663,0.295795,0.132452,0.308706,-0.596144,-0.370036,-0.288416
5,FedSCDepth(Average&Retrain),0.226523,0.092260,0.317537,0.160929,0.328214,-0.593822,-0.370802,-0.286740
6,FedSCDepth(Average),0.173326,-0.153443,0.286916,0.108006,0.302826,-0.593817,-0.372213,-0.289697
7,FedSCDepth(Retrain),0.272264,0.281354,0.340981,0.212965,0.346543,-0.587747,-0.373780,-0.289859
3,BOFedSCDepth(Retrain),0.093695,-0.052885,0.149558,0.076739,0.126917,-0.263837,-0.076983,-0.028170
0,BOFedSCDepth,0.089440,-0.132339,0.149636,0.048332,0.116336,-0.259735,-0.072052,-0.025753
2,BOFedSCDepth(ConstrainedLossRetrain),0.059228,-0.211707,0.133393,0.039766,0.111008,-0.258484,-0.073650,-0.026970
1,BOFedSCDepth(ConstrainedLoss),0.036870,-0.266628,0.111051,0.016661,0.088910,-0.246638,-0.068955,-0.023159
8,SCDepth,-0.215681,-0.937468,-0.156977,-0.166340,-0.112490,-0.125886,-0.022880,-0.003248


In [13]:
df_diff_prop_avg_ciclic.sort_values(by=['test_a2'],ascending=True)

,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
7,FedSCDepth(Retrain),0.272264,0.281354,0.340981,0.212965,0.346543,-0.587747,-0.373780,-0.289859
6,FedSCDepth(Average),0.173326,-0.153443,0.286916,0.108006,0.302826,-0.593817,-0.372213,-0.289697
5,FedSCDepth(Average&Retrain),0.226523,0.092260,0.317537,0.160929,0.328214,-0.593822,-0.370802,-0.286740
4,FedSCDepth,0.190863,-0.038663,0.295795,0.132452,0.308706,-0.596144,-0.370036,-0.288416
3,BOFedSCDepth(Retrain),0.093695,-0.052885,0.149558,0.076739,0.126917,-0.263837,-0.076983,-0.028170
2,BOFedSCDepth(ConstrainedLossRetrain),0.059228,-0.211707,0.133393,0.039766,0.111008,-0.258484,-0.073650,-0.026970
0,BOFedSCDepth,0.089440,-0.132339,0.149636,0.048332,0.116336,-0.259735,-0.072052,-0.025753
1,BOFedSCDepth(ConstrainedLoss),0.036870,-0.266628,0.111051,0.016661,0.088910,-0.246638,-0.068955,-0.023159
8,SCDepth,-0.215681,-0.937468,-0.156977,-0.166340,-0.112490,-0.125886,-0.022880,-0.003248


In [14]:
df_diff_prop_avg_ciclic.sort_values(by=['test_a3'],ascending=True)

,training_method,test_abs_rel,test_sq_rel,test_log10,test_rmse,test_rmse_log,test_a1,test_a2,test_a3
7,FedSCDepth(Retrain),0.272264,0.281354,0.340981,0.212965,0.346543,-0.587747,-0.373780,-0.289859
6,FedSCDepth(Average),0.173326,-0.153443,0.286916,0.108006,0.302826,-0.593817,-0.372213,-0.289697
4,FedSCDepth,0.190863,-0.038663,0.295795,0.132452,0.308706,-0.596144,-0.370036,-0.288416
5,FedSCDepth(Average&Retrain),0.226523,0.092260,0.317537,0.160929,0.328214,-0.593822,-0.370802,-0.286740
3,BOFedSCDepth(Retrain),0.093695,-0.052885,0.149558,0.076739,0.126917,-0.263837,-0.076983,-0.028170
2,BOFedSCDepth(ConstrainedLossRetrain),0.059228,-0.211707,0.133393,0.039766,0.111008,-0.258484,-0.073650,-0.026970
0,BOFedSCDepth,0.089440,-0.132339,0.149636,0.048332,0.116336,-0.259735,-0.072052,-0.025753
1,BOFedSCDepth(ConstrainedLoss),0.036870,-0.266628,0.111051,0.016661,0.088910,-0.246638,-0.068955,-0.023159
8,SCDepth,-0.215681,-0.937468,-0.156977,-0.166340,-0.112490,-0.125886,-0.022880,-0.003248
